In [3]:
import pandas as pd

cv = pd.read_parquet("recruitment-dataset-candidate-profiles-english.parquet")
job = pd.read_parquet("recruitment-dataset-job-descriptions-english.parquet")

# bỏ cột không cần
cv = cv.drop(columns=["CV_lang", "__index_level_0__"], errors="ignore")
job = job.drop(columns=["Long Description_lang", "__index_level_0__"], errors="ignore")

# thay NaN bằng chuỗi rỗng
cv = cv.fillna("")
job = job.fillna("")

# hàm làm sạch text nhẹ
def clean_text(text):
    text = str(text)
    text = text.replace("\\r\\n", " ")
    text = text.replace("\r\n", " ")
    text = text.replace("\n", " ")
    text = text.replace("\r", " ")
    text = " ".join(text.split())
    return text.strip()

# clean từng cột text của CV
cv["Position"] = cv["Position"].apply(clean_text)
cv["Moreinfo"] = cv["Moreinfo"].apply(clean_text)
cv["Looking For"] = cv["Looking For"].apply(clean_text)
cv["Highlights"] = cv["Highlights"].apply(clean_text)
cv["Primary Keyword"] = cv["Primary Keyword"].apply(clean_text)
cv["CV"] = cv["CV"].apply(clean_text)

# clean từng cột text của Job
job["Position"] = job["Position"].apply(clean_text)
job["Long Description"] = job["Long Description"].apply(clean_text)
job["Primary Keyword"] = job["Primary Keyword"].apply(clean_text)
job["Exp Years"] = job["Exp Years"].apply(clean_text)
job["English Level"] = job["English Level"].apply(clean_text)

# giữ text gốc bên CV
cv["cv_text_raw"] = cv["CV"]

# tự gộp text CV
cv["cv_text"] = (
    cv["Position"] + " " +
    cv["Moreinfo"] + " " +
    cv["Looking For"] + " " +
    cv["Highlights"] + " " +
    cv["Primary Keyword"]
).apply(clean_text)

# tự gộp text Job
job["job_text"] = (
    job["Position"] + " " +
    job["Long Description"] + " " +
    job["Primary Keyword"] + " " +
    job["Exp Years"] + " " +
    job["English Level"]
).apply(clean_text)

# bỏ dòng rỗng ở text chính
cv = cv[cv["cv_text"].str.len() > 0].copy()
job = job[job["job_text"].str.len() > 0].copy()

print("CV shape:", cv.shape)
print("JOB shape:", job.shape)


CV shape: (210250, 11)
JOB shape: (141897, 9)


In [4]:
cv[["id", "cv_text_raw", "cv_text"]].head()

,id,cv_text_raw,cv_text
0,50534b61-6826-52b1-9ac5-bfd2cfa348ec,Landed a role of Director of Blockchain Develo...,"13 years of exp || Solidity, C#, JavaScript ||..."
1,c2b9ea56-b5c8-50ec-a63b-053a5c8ff467,Worked on a mobile application for tracking trips,1c Developer Worked on a mobile application fo...
2,b3bfe3ed-ec25-56b2-8aaa-8c629120538e,1 am an 1C developer. I deployed an 1C to typo...,1C developer 1 am an 1C developer. I deployed ...
3,163fb9a3-3695-5dc3-b1d0-e0038ce4d0a5,Perfect knowledge of 1C:Enterprise Platform. G...,1C Developer Perfect knowledge of 1C:Enterpris...
4,3f201183-db58-5333-9139-74c16059dc4a,"thousands of resolved problems, also, thousand...","#1 Customer Support Specialist Tech addicted, ..."


In [5]:
job[["id", "job_text"]].head()

,id,job_text
0,c0ca96e7-85df-50df-a64e-d934cd02a170,10 + Blockchain Nodes / Masternodes to set up ...
1,64f4b7ea-36e4-5bdd-a8b1-185f32f7dc7f,10 .NET Developers (Middle and Senior level) G...
2,b9a1303e-dd0c-5ed1-8f62-be2bc4c7da4f,"10X Engineer (co-founder, #4 employee, USD 11-..."
3,99cb3f4a-9b4b-53d9-9a3b-bab2c22da346,"16 - Amazon Brand Manager Currently, TCM expan..."
4,bc1419f7-28e2-582b-8d53-22e28b2f0210,"16 - Amazon Brand Manager Hello, We, MIMIRB2B,..."


In [6]:
cv["cv_text_len"] = cv["cv_text"].str.len()
job["job_text_len"] = job["job_text"].str.len()

cv["cv_text_len"].describe()

count    210250.000000
mean        958.282730
std         667.173659
min          19.000000
25%         476.000000
50%         768.000000
75%        1245.000000
max        7366.000000
Name: cv_text_len, dtype: float64

In [7]:
job["job_text_len"].describe()

count    141897.000000
mean       1809.444202
std         959.911278
min          97.000000
25%        1127.000000
50%        1639.000000
75%        2286.000000
max       12416.000000
Name: job_text_len, dtype: float64

In [8]:
print("CV text rỗng:", (cv["cv_text"].str.len() == 0).sum())
print("JOB text rỗng:", (job["job_text"].str.len() == 0).sum())

CV text rỗng: 0
JOB text rỗng: 0


In [9]:
cv[["id", "cv_text"]].sample(5, random_state=42)

,id,cv_text
207440,0cef867b-fd31-5946-9604-1055cad5d45e,Разработчик java February 2017 – February 2022...
36952,dc5a1f77-c8b2-53db-8a3c-698ff8bff462,Front-end developer I'm looking for my first j...
16095,33b75dcc-9342-5be3-9a7b-fe8b6213cf06,C# .NET Core Developer My Experience: Made a p...
41590,aebacbbf-ae5f-5321-b7c1-fb2d31903127,Front End Developer Building react application...
85341,8a7cb9d4-938f-54c7-b8f7-9c5a5a7e90eb,"Junior Java Developer OOP, AOP (basics) Java (..."


In [10]:
job[["id", "job_text"]].sample(5, random_state=42)

,id,job_text
138255,e7f4270d-5f9d-5731-9572-a1a7cd9d9f46,Trainee Visual / Interaction Designer **Projec...
29788,a0c54df6-2d88-5d08-9a71-6b4081135a10,Expert Java Developer for Ciklum Digital Proje...
140542,12562761-a160-519e-a744-d0b306d51186,Web Analyst 42DM is a digital marketing agency...
96207,60ae7553-6deb-58eb-b330-aa3f7848040d,QA Automation/ Manual QA Engineer The Ideal Ca...
118230,7a9fd3e3-e2a2-5a70-a7d1-27a46721933e,Senior JS (React) Engineer Hey Engineers! 🙂 Th...
